# Mapa

Mapa ČR zobrazující obce/volební okrsky a jejich podíl prefenčních hlasů pro ženy. 
Na úrovni volebních okrsků jsou výsledky vidět i pro jednotlivé městské části, pokud je města mají. 

In [1]:
import geopandas as gpd

okrsky = gpd.read_file("vol_okrsky_2025_20250701.geojson")
hranice_cr = okrsky.dissolve()
hranice_cr.to_file("hranice_cr.geojson", driver="GeoJSON")
print("Hotovo!")

Hotovo!


In [7]:
# mapa obce
import folium
import pandas as pd

obce = pd.read_csv("centroidy_obce.csv", sep=",", encoding="utf-8-sig")
obce = obce.rename(columns={"KodObce": "KodObec"})

kvartaly = pd.read_csv("rozdeleni_okrsku_kvartaly.csv")
kvartaly["PROCPOC"] = (kvartaly["PROCPOC"] * 100).round(decimals=2)

obce_kvartaly = kvartaly.merge(obce, on="KodObec", how="left")

barvy = {
    1: {'color': '#C1121F', 'opacity': 0.3},
    2: {'color': '#B50D11', 'opacity': 0.5},
    3: {'color': '#C1121F', 'opacity': 0.7},
    4: {'color': '#780000', 'opacity': 1.0},
}
obce_kvartaly['barva'] = obce_kvartaly['KVARTAL'].map(barvy)

# mapa
mapa = folium.Map(location=[49.8, 15.5], zoom_start=8, tiles=None)

# hranice ČR
folium.GeoJson(
    "hranice_cr.geojson",
    style_function=lambda x: {
        "fillColor": "#E9ECEB",
        "color": "#003049",
        "weight": 2,
        "fillOpacity": 1
    }
).add_to(mapa)

# body
for _, radek in obce_kvartaly.iterrows():
    folium.CircleMarker(
        location=[radek["lat"], radek["lon"]],
        radius=3,
        color=barvy[radek["KVARTAL"]]['color'],
        weight=0,
        fill=True,
        fill_color=barvy[radek["KVARTAL"]]['color'],
        fill_opacity=barvy[radek["KVARTAL"]]['opacity'],
        tooltip=f"{radek['NAZEV']}<br>Podíl prefenčních hlasů pro ženy: {radek['PROCPOC']}%"
    ).add_to(mapa)

legenda_html = """
<div style="position: fixed; bottom: 30px; left: 30px; z-index: 1000; 
     background-color: #E9ECEB; padding: 15px; border-radius: 8px;
     border: 2px solid #003049; font-family: Arial; color: #003049;">
    <b>Kvartil podílu preferenčních hlasů pro ženy</b><br><br>
    <i style="background:#C1121F; opacity:0.2; width:15px; height:15px; 
       display:inline-block; border-radius:50%;"></i>&nbsp; Kvartil 1 (0,0 – 24,77 %)<br>
    <i style="background:#B50D11; opacity:0.5; width:15px; height:15px; 
       display:inline-block; border-radius:50%;"></i>&nbsp; Kvartil 2 (24,77 – 30,30 %)<br>
    <i style="background:#C1121F; opacity:0.7; width:15px; height:15px; 
       display:inline-block; border-radius:50%;"></i>&nbsp; Kvartil 3 (30,30 – 36,04 %)<br>
    <i style="background:#780000; opacity:1.0; width:15px; height:15px; 
       display:inline-block; border-radius:50%;"></i>&nbsp; Kvartil 4 (36,04 – 100,0 %)<br>
</div>
"""
mapa.get_root().html.add_child(folium.Element(legenda_html))

mapa.save("mapa_obce.html")
print("Hotovo!")

Hotovo!


In [6]:
# mapa okrsky

import folium
import pandas as pd

okrsky = pd.read_csv("okrsky_geojson.csv", sep=";") # načíst okrsky - s geodaty

kvartaly = pd.read_csv("rozdeleni_okrsku_kvartaly.csv") # načíst kvartály
kvartaly["PROCPOC"] = (kvartaly["PROCPOC"] * 100).round(decimals=2) 

okrsky_kvartaly = okrsky.merge(kvartaly, on="IdOkrsek", how="left") # spojení oksků s geo a okrsků s kvartály


barvy = {
    1: {'color': '#C1121F', 'opacity': 0.2},
    2: {'color': '#B50D11', 'opacity': 0.5},
    3: {'color': '#C1121F', 'opacity': 0.7},
    4: {'color': '#780000', 'opacity': 1.0},
}
okrsky_kvartaly['barva'] = okrsky_kvartaly['KVARTAL'].map(barvy)

# mapa
mapa = folium.Map(location=[49.8, 15.5], zoom_start=8, tiles=None)

# hranice ČR
folium.GeoJson(
    "hranice_cr.geojson",
    style_function=lambda x: {
        "fillColor": "#E9ECEB",
        "color": "#003049",
        "weight": 2,
        "fillOpacity": 1
    }
).add_to(mapa)

# body
for _, radek in okrsky_kvartaly.iterrows():
    nazev_mco = radek["naz_mco"]
    nazev_obec = radek["naz_obec"]
    mco = ("<br>" + nazev_mco) if nazev_mco.strip() != "" else "<br>-"
    procpoc = radek["PROCPOC"]
    tooltip = f"{nazev_obec}{mco}<br>Podíl preferenčních hlasů pro ženy: {procpoc}%"
    folium.CircleMarker(
        location=[radek["lat"], radek["lon"]],
        radius=3,
        color=barvy[radek["KVARTAL"]]['color'],
        weight=0,
        fill=True,
        fill_color=barvy[radek["KVARTAL"]]['color'],
        fill_opacity=barvy[radek["KVARTAL"]]['opacity'],
        tooltip=tooltip
    ).add_to(mapa)

legenda_html = """
<div style="position: fixed; bottom: 30px; left: 30px; z-index: 1000; 
     background-color: #E9ECEB; padding: 15px; border-radius: 8px;
     border: 2px solid #003049; font-family: Arial; color: #003049;">
    <b>Kvartil podílu preferenčních hlasů pro ženy</b><br><br>
    <i style="background:#C1121F; opacity:0.2; width:15px; height:15px; 
       display:inline-block; border-radius:50%;"></i>&nbsp; Kvartil 1 (0,0 – 24,77 %)<br>
    <i style="background:#B50D11; opacity:0.5; width:15px; height:15px; 
       display:inline-block; border-radius:50%;"></i>&nbsp; Kvartil 2 (24,77 – 30,30 %)<br>
    <i style="background:#C1121F; opacity:0.7; width:15px; height:15px; 
       display:inline-block; border-radius:50%;"></i>&nbsp; Kvartil 3 (30,30 – 36,04 %)<br>
    <i style="background:#780000; opacity:1.0; width:15px; height:15px; 
       display:inline-block; border-radius:50%;"></i>&nbsp; Kvartil 4 (36,04 – 100,0 %)<br>
</div>
"""
mapa.get_root().html.add_child(folium.Element(legenda_html))

mapa.save("mapa_okrsky.html")
print("Hotovo!")

Hotovo!
